# Build the mosaics: a development run, or the production run

This notebook has **two modes**. They are not variations on each other. They
differ in what gets built, where it is written, and how much it costs.

**`development` — one cell, one or a few years, into a collection you own.**
This is for trying something out, and for checking that the pipeline works
after you clone it. You pick a grid cell and a year, and the notebook queues
that one mosaic. A handful of tasks, minutes to set up. **This is the
default.**

**`production` — every cell, every year, into the published collection.**
The real thing: all 283 grid cells of India for every year from 1986 to 2025,
written into the MapBiomas collection that people will use. **11,320 export
tasks.** It costs a large amount of compute, takes days to work through, and
needs write access to a collection you must already own. You cannot start it
by accident: you have to switch the mode *and* paste a confirmation phrase.

A "year" in both modes is the phenological year: 1 April of the labelled year
to 31 March of the next. So `2019` means April 2019 to March 2020.

Both modes use the same pipeline code that builds the product. Nothing here
is a shortcut or a simplified copy.

## Before you start, if you have just cloned this

You need three things.

1. **Python packages.** `earthengine-api` and `jupyter`. The pipeline needs
   nothing else to queue an export; the work happens on Google's servers.
2. **An Earth Engine account, authenticated.** Run `earthengine authenticate`
   once in a terminal.
3. **A cloud project you can use**, and for a development run, somewhere you
   can write. Put your own project in `EE_PROJECT` below. The notebook makes
   the destination collection for you if it does not exist yet.

You do **not** need access to this project's own sandbox, and you do not need
write access to the published collection unless you are doing the production
run.

**How to use this notebook:** set the parameters in the next cell, run the
setup cell, then run the cells under the heading for your mode. The cells for
the other mode do nothing when the mode is not selected, so it is safe to run
the whole notebook from top to bottom.

In [ ]:
# ---------------------------------------------------------------------------
# Parameters -- the only cell you normally edit
# ---------------------------------------------------------------------------

# 'development' -> one cell, the years listed below, into a collection you own
# 'production'  -> every cell, every year, into the published collection
MODE = 'development'

# The Earth Engine cloud project to run under. Leave as '' to use the one in
# pipeline/config.py, which only works if you are a member of it. If you have
# cloned this repository and are running it from your own account, put your
# own project id here, for example 'my-ee-project'.
EE_PROJECT = ''

# --- development mode only -------------------------------------------------
CELL = 'NC-43-X-D'          # grid cell name (must exist in the CIM grid asset)
YEARS = [2019]              # one or more phenological years, e.g. [2018, 2019]

# Where the trial mosaic is written. Leave as '' to use this project's own
# sandbox, which you can only write to if you are a member of it. Otherwise
# give a collection under your own project; it is created if it is not there,
# for example 'projects/my-ee-project/assets/ioln_dev'.
DEV_COLLECTION = ''

# --- production mode only --------------------------------------------------
# The words below must be pasted in exactly before anything is queued. They
# are the only thing standing between a stray Shift+Enter and 11,320 tasks.
CONFIRM = ''                # to run: CONFIRM = 'QUEUE THE NATIONAL RUN'

# You do not need to set this. Earth Engine will only hold so many pending
# tasks at once, and how many depends on the account, so the run watches for
# the refusal and stops the sitting cleanly by itself, saying how far it got.
# Let the queue drain, run the cell again, and it carries on: already exported
# and currently exporting cell-years are skipped, never repeated.
# Set a number only if you want a deliberately short sitting.
MAX_TASKS = None

In [ ]:
# ---------------------------------------------------------------------------
# Setup: Earth Engine, pipeline imports, and a plain statement of what mode
# you are in and where things will be written
# ---------------------------------------------------------------------------

import os
import sys

# The notebook lives in notebooks/; the pipeline package lives one level up.
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.isdir(os.path.join(repo_root, 'pipeline')) and repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import ee

from pipeline import config
from pipeline import build
from pipeline import run as run_module
from pipeline import run_production

project = EE_PROJECT or config.EE_PROJECT
ee.Initialize(project=project)
print('Earth Engine project: {}'.format(project))

# Same check the batch drivers run: loud about any coefficient set not yet
# marked verified. Warnings do not block a run.
run_module.preflight()

if MODE == 'development':
    # Deliberately NOT config.OUTPUT_COLLECTION. In the published fork that
    # constant points at the production collection, so a development run left
    # on its default would aim at the real product. A trial build goes to a
    # sandbox, or to a collection you named yourself.
    COLLECTION_PATH = DEV_COLLECTION or config.SANDBOX_COLLECTION
    if COLLECTION_PATH == config.PRODUCTION_COLLECTION:
        raise ValueError(
            'a development run must not write to the production collection. '
            'Set DEV_COLLECTION to somewhere you own.')
    print('\nMODE: development -- a trial build, nothing published')
    print('  building:    {} for year(s) {}'.format(CELL, YEARS))
    print('  writing to:  {}'.format(COLLECTION_PATH))
    print('  asset names: <CELL>_<year>_c2_only_v{}'.format(config.VERSION))
    print('  the collection is created for you if it does not exist')
elif MODE == 'production':
    COLLECTION_PATH = config.PRODUCTION_COLLECTION
    print('\nMODE: production -- THE REAL RUN, into the published collection')
    print('  building:    every grid cell, {}..{}'.format(
        config.PRODUCTION_FIRST_YEAR, config.PRODUCTION_LAST_YEAR))
    print('  writing to:  {}'.format(COLLECTION_PATH))
    print('  asset names: <CELL>_<year>')
    try:
        ee.data.getAsset(COLLECTION_PATH)
        print('  destination reachable: yes')
    except Exception as e:
        print('  destination reachable: NO -- {}'.format(str(e)[:110]))
        print('  You need write access to that collection before the run can')
        print('  do anything. Sort that out first, or the run will fail on')
        print('  every cell-year.')
else:
    raise ValueError("MODE must be 'development' or 'production', "
                     'got {!r}'.format(MODE))

## Development run

One cell, the years you listed, into the sandbox. The cell below does nothing
unless `MODE` is `'development'`.

Re-running is safe. A cell-year already exported, or already being exported,
is skipped rather than queued twice. A cell-year with no usable satellite
imagery (common before 2000 in some regions) is skipped with a message rather
than exported empty: the gap is real, and the honest product shows it.

In [ ]:
if MODE != 'development':
    print('skipped: MODE is {!r}, not "development"'.format(MODE))
else:
    tasks = []
    for year in YEARS:
        task = build.export(CELL, year, variant='c2_only',
                            collection_path=COLLECTION_PATH, verbose=True)
        if task is not None:
            tasks.append(task)

    print('\n{} task(s) queued to {}'.format(len(tasks), COLLECTION_PATH))
    for task in tasks:
        print('  {}'.format(task.config['description']))

## Production run

Every grid cell, every year, into the published collection. Two cells: the
first only tells you what would happen, the second does it.

**Read the plan before you run the second cell.** Once tasks are queued they
consume compute until they finish or you cancel them one by one.

In [ ]:
# The plan. Nothing is queued by this cell.

if MODE != 'production':
    print('skipped: MODE is {!r}, not "production"'.format(MODE))
else:
    work = run_production.plan()
    cells = sorted({c for c, _ in work})
    years = sorted({y for _, y in work})
    print('{} cells x {} years = {} cell-years'.format(
        len(cells), len(years), len(work)))
    print('years        {}..{}'.format(years[0], years[-1]))
    print('destination  {}'.format(COLLECTION_PATH))
    print('this sitting stops after {} queued task(s)'.format(MAX_TASKS)
          if MAX_TASKS else 'no limit on tasks this sitting')
    print('\nfirst few: {}'.format(work[:3]))
    print('last few:  {}'.format(work[-3:]))

In [ ]:
# The run itself. Needs MODE = 'production' AND the confirmation phrase.

if MODE != 'production':
    print('skipped: MODE is {!r}, not "production"'.format(MODE))
elif CONFIRM != run_production.CONFIRM_PHRASE:
    print('skipped: nothing was queued.')
    print('To start the national run, set CONFIRM in the parameters cell to')
    print('    {!r}'.format(run_production.CONFIRM_PHRASE))
else:
    summary = run_production.run(confirm=CONFIRM, max_tasks=MAX_TASKS)

## Track the run

Safe to run at any time, in either mode, as often as you like. It queues
nothing and changes nothing: it counts what has actually landed in the
collection, what Earth Engine is working on now, and what has failed.

For the production run this is the cell to come back to. It is the answer to
"how far along are we", and it works after you have closed and reopened the
notebook, because it reads the state from Earth Engine rather than from
anything this notebook remembers.

In [ ]:
try:
    if MODE == 'development':
        state = run_production.progress(collection_path=COLLECTION_PATH,
                                        cells=[CELL],
                                        first_year=min(YEARS),
                                        last_year=max(YEARS))
    else:
        state = run_production.progress(
            collection_path=COLLECTION_PATH,
            keep_failures_in='run_failures.log')
except RuntimeError as e:
    # Nearly always one thing: you cannot read the destination collection,
    # so there is nothing to count yet.
    state = None
    print('cannot report progress yet.')
    print(e)

## What happens next

Each queued task runs on Google's servers -- you can close this notebook and
come back. Use the tracking cell above to see how far along things are, or
watch the **Tasks** tab of the
[Earth Engine Code Editor](https://code.earthengine.google.com/).

When a task finishes, the mosaic appears inside the collection printed by the
setup cell.

For the production run, expect to come back several times. Earth Engine will
not hold 11,320 pending tasks at once. You do not have to know its limit: when
it refuses more, the run stops the sitting, says how many it queued and how far
it reached, and waits. Let the queue drain, run the queueing cell again, and it
carries on. Re-running never duplicates work, because a cell-year that is
already exported or already being exported is skipped.

If many cell-years fail one after another, the run stops itself and says so.
That means something is wrong with the set-up rather than with one cell:
usually credentials, or no write access to the destination. Fix the cause and
start again; nothing already queued is lost.

The same run can be driven from a terminal instead, which is steadier for
something that takes days:

```
python -m pipeline.run_production --plan
python -m pipeline.run_production --run --confirm "QUEUE THE NATIONAL RUN" --max-tasks 500
python -m pipeline.run_production --progress
```